# Inventory Aging Analysis


## Project Overview

This notebook analyses 800 inventory items to identify aging stock, quantify value at risk, classify slow movers, and assess reorder-point compliance.


## Learning Objectives

- Build aging buckets and compute inventory value by bucket
- Identify slow-moving and obsolete SKUs
- Analyse ABC class distribution relative to aging
- Assess reorder point compliance by category


## Business or Research Problem

Excess and aging inventory ties up working capital and incurs storage costs. Inventory managers need a systematic view of which SKUs are aging and where value is at risk.


## Why This Analysis Matters

Reducing aged inventory improves cash flow, reduces write-off risk, and frees warehouse space for faster-moving products.


## Dataset Overview

- **Rows:** 800 inventory items
- **Key columns:** sku, category, sub_category, units_in_stock, days_in_stock, cost_per_unit, last_sold_days_ago, reorder_point, abc_class


## Dataset Source and License Notes

Fully synthetic data generated with NumPy seed=42. For educational use only.


## Environment Setup


In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('Python:', sys.version)


## Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Configuration / Constants


In [ ]:
SEED = 42
N = 800
CATEGORIES = ['Electronics', 'Apparel', 'Home Goods', 'Food & Bev', 'Industrial']
SUB_CATS = {'Electronics': ['Phones', 'Laptops', 'Accessories'],
            'Apparel': ['Shirts', 'Pants', 'Footwear'],
            'Home Goods': ['Furniture', 'Decor', 'Kitchen'],
            'Food & Bev': ['Dry Goods', 'Beverages', 'Snacks'],
            'Industrial': ['Tools', 'Safety', 'Parts']}
AGING_BINS = [0, 30, 90, 180, float('inf')]
AGING_LABELS = ['0-30 days', '31-90 days', '91-180 days', '181+ days']


## Dataset Download or Loading


In [ ]:
rng = np.random.default_rng(SEED)
categories = rng.choice(CATEGORIES, size=N)
sub_cats = [rng.choice(SUB_CATS[c]) for c in categories]
skus = [f'SKU{str(i).zfill(4)}' for i in range(1, N+1)]
units_in_stock = rng.integers(0, 500, N)
days_in_stock = rng.integers(1, 400, N)
cost_per_unit = rng.uniform(2, 500, N).round(2)
last_sold_days_ago = days_in_stock + rng.integers(-10, 30, N)
last_sold_days_ago = np.clip(last_sold_days_ago, 0, None)
reorder_point = rng.integers(10, 100, N)
abc_class = rng.choice(['A', 'B', 'C'], size=N, p=[0.2, 0.3, 0.5])

df = pd.DataFrame({'sku': skus, 'category': categories, 'sub_category': sub_cats,
    'units_in_stock': units_in_stock, 'days_in_stock': days_in_stock,
    'cost_per_unit': cost_per_unit, 'last_sold_days_ago': last_sold_days_ago,
    'reorder_point': reorder_point, 'abc_class': abc_class})
df['inventory_value'] = df['units_in_stock'] * df['cost_per_unit']
df['aging_bucket'] = pd.cut(df['days_in_stock'], bins=AGING_BINS, labels=AGING_LABELS)
df['below_reorder'] = df['units_in_stock'] < df['reorder_point']
print(df.shape)
df.head(3)


## Data Validation Checks


In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nAging bucket counts:')
print(df['aging_bucket'].value_counts().sort_index())
assert df['units_in_stock'].min() >= 0
print('All validation checks passed.')


## Data Cleaning


In [ ]:
print('Duplicate SKUs:', df['sku'].duplicated().sum())
df = df.drop_duplicates(subset='sku')
print('Clean shape:', df.shape)


## Exploratory Data Analysis


In [ ]:
print(df[['units_in_stock','days_in_stock','cost_per_unit','inventory_value']].describe().round(2))
total_value = df['inventory_value'].sum()
print(f'\nTotal inventory value: ${total_value:,.0f}')


## Univariate Analysis

Distribution of days in stock and inventory value per SKU.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df['days_in_stock'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Days in Stock Distribution')
axes[0].set_xlabel('Days')
axes[1].hist(df['inventory_value'], bins=30, color='salmon', edgecolor='white')
axes[1].set_title('Inventory Value per SKU')
axes[1].set_xlabel('Value ($)')
plt.tight_layout()
plt.show()


Most SKUs have been in stock under 200 days, with a long tail indicating potential obsolescence.


## Bivariate / Multivariate Analysis

Inventory value at risk by aging bucket and category.


In [ ]:
bucket_value = df.groupby('aging_bucket', observed=True)['inventory_value'].sum().reset_index()
bucket_value['pct'] = bucket_value['inventory_value'] / total_value * 100
print(bucket_value.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(bucket_value['aging_bucket'].astype(str), bucket_value['inventory_value'] / 1e6, color='coral')
ax.set_title('Inventory Value at Risk by Aging Bucket')
ax.set_ylabel('Value ($M)')
plt.tight_layout()
plt.show()


## Feature-Specific Insights

ABC class distribution across aging buckets.


In [ ]:
abc_aging = df.groupby(['aging_bucket', 'abc_class'], observed=True).size().unstack(fill_value=0)
abc_aging.plot(kind='bar', figsize=(10, 4), colormap='Set2')
plt.title('SKU Count by Aging Bucket and ABC Class')
plt.ylabel('SKU Count')
plt.legend(title='ABC Class')
plt.tight_layout()
plt.show()
old_a_class = df[(df['aging_bucket'] == '181+ days') & (df['abc_class'] == 'A')].shape[0]
print(f'High-value (Class A) SKUs aged 181+ days: {old_a_class}')


## Statistical Checks

Kruskal-Wallis test — does inventory value differ across aging buckets?


In [ ]:
groups = [g['inventory_value'].values for _, g in df.groupby('aging_bucket', observed=True)]
h_stat, p_val = stats.kruskal(*groups)
print(f'Kruskal-Wallis H={h_stat:.2f}, p={p_val:.4f}')
if p_val < 0.05:
    print('Inventory value distribution differs significantly across aging buckets.')


## Visual Analysis

Category-level aging breakdown as a stacked bar.


In [ ]:
cat_bucket = df.groupby(['category', 'aging_bucket'], observed=True)['inventory_value'].sum().unstack(fill_value=0) / 1e3
cat_bucket.plot(kind='bar', stacked=True, figsize=(11, 5), colormap='RdYlGn_r')
plt.title('Inventory Value ($K) by Category and Aging Bucket')
plt.ylabel('Value ($K)')
plt.legend(title='Aging Bucket', bbox_to_anchor=(1.05, 1))
plt.tight_layout()
plt.show()


## Slow-Mover Identification


In [ ]:
slow_movers = df[df['last_sold_days_ago'] > 120].copy()
print(f'Slow-moving SKUs (last sold > 120 days ago): {len(slow_movers)}')
slow_value = slow_movers['inventory_value'].sum()
print(f'Value tied up in slow movers: ${slow_value:,.0f} ({slow_value/total_value:.1%} of total)')
print('\nTop 5 slow movers by value:')
print(slow_movers.nlargest(5, 'inventory_value')[['sku','category','units_in_stock','inventory_value','last_sold_days_ago']].to_string(index=False))


## Key Findings


In [ ]:
aged_pct = (df['aging_bucket'] == '181+ days').mean()
below_ro = df['below_reorder'].mean()
print(f'1. {aged_pct:.1%} of SKUs have been in stock over 181 days')
print(f'2. Class-A SKUs aged 181+ days: {old_a_class} (high-priority write-down candidates)')
print(f'3. Slow-moving SKUs represent ${slow_value:,.0f} ({slow_value/total_value:.1%}) of total inventory value')
print(f'4. {below_ro:.1%} of SKUs are below their reorder point — replenishment needed')


## Limitations

- Synthetic data; real demand patterns may vary seasonally
- Reorder points are randomly assigned, not demand-derived
- No forecast data to distinguish truly obsolete vs slow-seasonal items


## Recommendations / Next Steps

1. Initiate markdown or clearance campaigns for 181+ day non-Class-A items
2. For Class-A slow movers, investigate root cause (pricing, placement, demand shift)
3. Automate reorder alerts for SKUs below reorder point
4. Revisit reorder points quarterly using actual demand data


## Common Mistakes

- Using only days_in_stock without considering units or value leads to over-representation of cheap items
- Ignoring ABC class: writing down Class-A slow movers needs more analysis than Class-C
- Treating all 181+ items as waste; some may be strategic safety stock


## Mini Challenge / Exercises

1. Create a `turnover_ratio` = units sold / avg stock and rank SKUs
2. Calculate total holding cost assuming 20% annual carrying cost rate
3. Identify categories where Class-C items dominate the 181+ bucket (low value, high space waste)


## Final Summary / Key Takeaways

- Aging buckets provide a clear tiered view of at-risk inventory
- ABC-aging cross-analysis prioritises which write-downs matter most financially
- Reorder compliance monitoring prevents stockouts while aging analysis prevents overstock
- Regular cadence (monthly) of this analysis is essential for working capital management
